In [1]:
import pandas as pd
from pathlib import Path

raw = Path("../data/raw")
orders = pd.read_csv(raw / "olist_orders_dataset.csv")
items = pd.read_csv(raw / "olist_order_items_dataset.csv")
customers = pd.read_csv(raw / "olist_customers_dataset.csv")
reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv")
products = pd.read_csv(raw / "olist_products_dataset.csv")

for name, df in [("orders", orders), ("items", items), ("customers", customers),
                 ("reviews", reviews), ("products", products)]:
    print(f"\n=== {name}: {df.shape} ===")
    print(df.isna().sum()[df.isna().sum() > 0])


=== orders: (99441, 8) ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

=== items: (112650, 7) ===
Series([], dtype: int64)

=== customers: (99441, 5) ===
Series([], dtype: int64)

=== reviews: (99224, 7) ===
review_comment_title      87656
review_comment_message    58247
dtype: int64

=== products: (32951, 9) ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


How many unique order_id vs unique customer_id vs unique customer_unique_id?

In [2]:
number_of_orders = orders.shape[0]
number_of_customers = customers.shape[0]
number_of_customers_unique = customers.customer_unique_id.nunique()

print(f"\nNumber of orders: {number_of_orders}")
print(f"Number of customers: {number_of_customers}")
print(f"Number of unique customers: {number_of_customers_unique}")



Number of orders: 99441
Number of customers: 99441
Number of unique customers: 96096


What are the distinct values of order_status? How many orders are delivered vs the rest?

In [3]:
status_counts = orders["order_status"].value_counts()

print("Distinct order_status values:", orders["order_status"].unique().tolist())
print(f"Delivered: {status_counts.get('delivered', 0)}")
print(f"All other statuses: {len(orders) - status_counts.get('delivered', 0)}")

Distinct order_status values: ['delivered', 'invoiced', 'shipped', 'processing', 'unavailable', 'canceled', 'created', 'approved']
Delivered: 96478
All other statuses: 2963


Which order timestamp columns have nulls, and why?

In [4]:
timestamp_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

for column in timestamp_columns:
    missing = orders[column].isna()
    print(f"\n=== {column}: {missing.sum()} missing ===")
    print(orders.loc[missing, "order_status"].value_counts())


=== order_approved_at: 160 missing ===
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

=== order_delivered_carrier_date: 1783 missing ===
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

=== order_delivered_customer_date: 2965 missing ===
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


Are there order_ids with more than one review in reviews?

In [5]:
reviews_per_order = reviews["order_id"].value_counts()
multiple_reviews = reviews_per_order[reviews_per_order > 1]

print(f"Order IDs with more than one review: {len(multiple_reviews)}")
display(multiple_reviews.rename("review_count").to_frame())

Order IDs with more than one review: 547


,review_count
order_id,
df56136b8031ecd28e200bb18e6ddb2e,3
c88b1d1b157a9999ce368f218a407141,3
03c939fd7fd3b38f8485a0f95798f1f6,3
8e17072ec97ce29f0e1f111e598b0c85,3
029863af4b968de1e5d6a82782e662f5,2
...,...
e00ed9d20c3479f9f0e9727ca9d60946,2
5c2049caca936695cfa43805ebe7d8e3,2
6066fbcebeaf378770ad9fbe8e30d1f0,2


How many products have a null category? Do all categories exist in the translation table?

In [6]:
category_translation = pd.read_csv(
    raw / "product_category_name_translation.csv"
)

null_category_count = products["product_category_name"].isna().sum()

product_categories = set(products["product_category_name"].dropna())
translated_categories = set(category_translation["product_category_name"])
missing_translations = sorted(product_categories - translated_categories)

print(f"Products with null category: {null_category_count}")
print(
    "All product categories exist in the translation table:",
    not missing_translations,
)
print("Missing translations:", missing_translations)

Products with null category: 610
All product categories exist in the translation table: False
Missing translations: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


What date range do the orders cover? Are there suspicious months at the edges?

In [7]:
purchase_dates = pd.to_datetime(orders["order_purchase_timestamp"])

orders["order_purchase_month"] = purchase_dates.dt.to_period("M")

print(
    "Date range:",
    purchase_dates.min(),
    "to",
    purchase_dates.max(),
)

monthly_counts = (
    orders.groupby("order_purchase_month")
    .size()
    .sort_index()
)

print("\nOrders per month:")
display(monthly_counts.to_frame("order_count"))

print("\nEdge months:")
display(monthly_counts.iloc[:3].to_frame("order_count"))
display(monthly_counts.iloc[-3:].to_frame("order_count"))

Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18

Orders per month:


,order_count
order_purchase_month,
2016-09,4
2016-10,324
2016-12,1
2017-01,800
2017-02,1780
2017-03,2682
2017-04,2404
2017-05,3700
2017-06,3245



Edge months:


,order_count
order_purchase_month,
2016-09,4
2016-10,324
2016-12,1


,order_count
order_purchase_month,
2018-08,6512
2018-09,16
2018-10,4


Does price * count of items per order match the payments total?